# K-Means on Sample Data
**by Group 3**

## Installing and Importing Required Libraries

Java is installed for PySpark

In [1]:
!sudo yum install -y java-11-amazon-corretto-headless

Last metadata expiration check: 1:27:50 ago on Sun Jul 26 06:31:28 2026.
Package java-11-amazon-corretto-headless-1:11.0.31+11-1.amzn2023.x86_64 is already installed.
Dependencies resolved.
Nothing to do.
Complete!


In [2]:
import os
import glob

# Search for the installed Java directory
java_paths = glob.glob('/usr/lib/jvm/java-11*')

if java_paths:
    # Dynamically set the environment variable to the found path
    os.environ["JAVA_HOME"] = java_paths[0]
    print(f"JAVA_HOME successfully set to: {os.environ['JAVA_HOME']}")
else:
    print("Java not found. Did the yum install command work?")

JAVA_HOME successfully set to: /usr/lib/jvm/java-11-amazon-corretto.x86_64


Importing the necessary libraries for the model training

In [3]:
import pandas as pd
import numpy as np
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score

Setup a SparkSession

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline
from pyspark import StorageLevel
from pyspark.ml.clustering import KMeans
from pyspark.ml.evaluation import ClusteringEvaluator

spark = (
    SparkSession.builder
    .appName("KMeans-Sample-Data")
    .master("local[*]")
    .config("spark.jars.packages",
            "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider",
            "com.amazonaws.auth.InstanceProfileCredentialsProvider")
    .config("spark.driver.memory", "10g")
    .getOrCreate()
)

INPUT_PATH = "s3a://dat204m-project-g3/cleaned_data_final/"
OUTPUT_PATH = "s3a://dat204m-project-g3/sampled_eda_data/"


print("Spark ready:", spark.version)

:: loading settings :: url = jar:file:/home/ec2-user/anaconda3/envs/pytorch/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ec2-user/.ivy2/cache
The jars for the packages stored in: /home/ec2-user/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0537353a-a1f5-4c74-bb7a-b521cbdc6f28;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 249ms :: artifacts dl 7ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|

26/07/26 07:59:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark ready: 3.3.0


Check if connection to s3 is stable

In [5]:
import boto3

s3 = boto3.client("s3")

try:
    print(s3.list_objects_v2(Bucket="dat204m-project-g3", MaxKeys=5))
except Exception as e:
    print(e)

{'ResponseMetadata': {'RequestId': '4X95X59FXHV8AXMX', 'HostId': 'GaH0c8yz66YdqImWLQbAybHoYlAtrezXL2ilmpNG31uANhGdbnPvnfAXXYOox4xEVsseptY5sCU=', 'HTTPStatusCode': 200, 'HTTPHeaders': {'x-amz-id-2': 'GaH0c8yz66YdqImWLQbAybHoYlAtrezXL2ilmpNG31uANhGdbnPvnfAXXYOox4xEVsseptY5sCU=', 'x-amz-request-id': '4X95X59FXHV8AXMX', 'date': 'Sun, 26 Jul 2026 07:59:26 GMT', 'x-amz-bucket-region': 'us-east-1', 'content-type': 'application/xml', 'transfer-encoding': 'chunked', 'server': 'AmazonS3'}, 'RetryAttempts': 0}, 'IsTruncated': True, 'Contents': [{'Key': 'athena-logs/Unsaved/2026/07/05/0948405c-ddc7-4308-a5ea-9e4150ecd730-manifest.csv', 'LastModified': datetime.datetime(2026, 7, 4, 16, 28, 45, tzinfo=tzlocal()), 'ETag': '"c90c48a2b97547599fca337a336b68a1"', 'ChecksumAlgorithm': ['SHA1'], 'ChecksumType': 'FULL_OBJECT', 'Size': 3240, 'StorageClass': 'STANDARD'}, {'Key': 'athena-logs/Unsaved/2026/07/05/0948405c-ddc7-4308-a5ea-9e4150ecd730.metadata', 'LastModified': datetime.datetime(2026, 7, 4, 16, 28

## Reading and Standardizing Sample Data

Read the Parquet file of the feature engineered sample data

In [6]:
EDA_DATA_PATH = "s3a://dat204m-project-g3/feature_engineered/"
features_df = spark.read.parquet(EDA_DATA_PATH)

26/07/26 07:59:26 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


Show the schema of the sample data

In [7]:
features_df.printSchema()

root
 |-- product_id: string (nullable = true)
 |-- views: long (nullable = true)
 |-- likes: long (nullable = true)
 |-- cart: long (nullable = true)
 |-- offers: long (nullable = true)
 |-- buy_start: long (nullable = true)
 |-- buy_comp: long (nullable = true)
 |-- unique_users: long (nullable = true)
 |-- unique_sessions: long (nullable = true)
 |-- avg_price: double (nullable = true)
 |-- brand_name: string (nullable = true)
 |-- category_path: string (nullable = true)
 |-- cond_good: long (nullable = true)
 |-- cond_new: long (nullable = true)
 |-- cond_like_new: long (nullable = true)
 |-- cond_fair: long (nullable = true)
 |-- cond_poor: long (nullable = true)
 |-- cond_unknown: long (nullable = true)
 |-- log_views: double (nullable = true)
 |-- log_likes: double (nullable = true)
 |-- log_cart: double (nullable = true)
 |-- log_offers: double (nullable = true)
 |-- log_buy_start: double (nullable = true)
 |-- log_buy_comp: double (nullable = true)
 |-- log_users: double (null

Show the first five lines of the sample data

In [8]:
features_df.show(5)

26/07/26 07:59:31 WARN package: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[Stage 1:>                                                          (0 + 1) / 1]

+----------+-----+-----+----+------+---------+--------+------------+---------------+------------------+----------+--------------------+---------+--------+-------------+---------+---------+------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+----------------+
|product_id|views|likes|cart|offers|buy_start|buy_comp|unique_users|unique_sessions|         avg_price|brand_name|       category_path|cond_good|cond_new|cond_like_new|cond_fair|cond_poor|cond_unknown|         log_views|         log_likes|          log_cart|        log_offers|     log_buy_start|      log_buy_comp|         log_users|      log_sessions|         log_price|     log_cond_good|      log_cond_new| log_cond_like_new|     log_cond_fair|     log_cond_poor|log_cond_unknown|
+----------+-----+-----+----+-

To reduce recomputation and improve performance, DataFrame is cached in memory and spills excess data to disk when needed

In [9]:
features_df = features_df.persist(StorageLevel.MEMORY_AND_DISK)

In [10]:
# Materialize cache
features_df.count()

449882

Frequency Encoding is applied for categorical columns 'brand_name' and 'category_path' 

In [11]:
freq_cols = [
    "brand_name",
    "category_path"
]

features_encoded = features_df

for c in freq_cols:
    freq = (
        features_df.groupBy(c)
                   .count()
                   .withColumnRenamed("count", f"{c}_freq")
    )

    features_encoded = (
        features_encoded
        .join(freq, on=c, how="left")
    )

All numerical columns are combined into one single feature vector using VectorAssembler

In [12]:
numeric_features = [
    "log_views",
    "log_likes",
    "log_cart",
    "log_offers",
    "log_buy_comp",
    "log_buy_start",
    "log_users",
    "log_sessions",
    "log_price",
    "brand_name_freq",
    "category_path_freq",
    "log_cond_good",
    "log_cond_new",
    "log_cond_like_new",
    "log_cond_fair",
    "log_cond_poor",
    "log_cond_unknown"
]

assembler = VectorAssembler(
    inputCols=numeric_features,
    outputCol="features_raw"
)

Standard scaling is applied to prevent features with larger numerical ranges from dominating the clustering process

In [13]:
scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=False,
    withStd=True
)

Pipeline is run to apply the vectorizing and standard scaling to the sample data

In [14]:
pipeline = Pipeline(stages=[
    assembler,
    scaler
])

In [15]:
pipeline_model = pipeline.fit(features_encoded)

In [16]:
kmeans_df = pipeline_model.transform(features_encoded)

## Initial Model Training

As we do not have any target data, we will be doing unsupervised modelling with KMeans

Initialize a KMeans model

In [17]:
kmeans = KMeans(
    featuresCol="features",
    predictionCol="cluster",
    k=10,
    seed=42
)

Fit the model to the data

In [18]:
model = kmeans.fit(kmeans_df)

Assign each product to its predicted cluster

In [19]:
clustered_products = model.transform(kmeans_df)

Evaluation Metrics used for KMeans are the following:

* Silhouette Score - Evaluates the similarity of an object to its own group relative to other groups, thereby reflecting the balance between cohesion and separation
* Davies-Bouldin Index - Quantifies group compactness and separation. A lower index signifies higher-quality clusters.
* Calinski–Harabasz Index - Used to compare different clustering results and help select the optimal number of clusters. A higher score indicates better-defined clusters with greater separation between clusters and higher compactness within clusters.

Based on Alshrouf, F., Masadeh, S., Al-Fraihat, D., & Al-Nsoor, R. (2025). Improving BIRCH hierarchical clustering algorithms for enhanced partitioning of medical data. Cluster Computing, 28(9), 600.

In [20]:
pdf = clustered_products.select("features", "cluster").toPandas()

X = np.vstack(pdf["features"].apply(lambda x: x.toArray()))
labels = pdf["cluster"].values

silhouette = silhouette_score(
    X,
    labels,
    sample_size=20000,
    random_state=42
)

dbi = davies_bouldin_score(X, labels)
chi = calinski_harabasz_score(X, labels)

print(f"Silhouette Score:        {silhouette:.4f}")
print(f"Davies-Bouldin Index:   {dbi:.4f}")
print(f"Calinski-Harabasz Index:{chi:.4f}")

Silhouette Score:        0.2042
Davies-Bouldin Index:   1.4075
Calinski-Harabasz Index:93783.6081


## Hyperparameter Tuning

We check which available hyperparameters can be changed

In [21]:
kmeans = KMeans()

print(kmeans.explainParams())

distanceMeasure: the distance measure. Supported options: 'euclidean' and 'cosine'. (default: euclidean)
featuresCol: features column name. (default: features)
initMode: The initialization algorithm. This can be either "random" to choose random points as initial cluster centers, or "k-means||" to use a parallel variant of k-means++ (default: k-means||)
initSteps: The number of steps for k-means|| initialization mode. Must be > 0. (default: 2)
k: The number of clusters to create. Must be > 1. (default: 2)
maxIter: max number of iterations (>= 0). (default: 20)
predictionCol: prediction column name. (default: prediction)
seed: random seed. (default: -8475663736230312989)
tol: the convergence tolerance for iterative algorithms (>= 0). (default: 0.0001)
weightCol: weight column name. If this is not set or empty, we treat all instance weights as 1.0. (undefined)


The selected hyperparameters to change are:
* K
* maxIter
* initMode
* initSteps

Based on Gikera, R., Mwaura, J., Muuro, E., & Mambo, S. K-Hyperparameter Tuning in High-Dimensional Space Clustering: Solving Smooth Elbow Challenges Using an Ensemble Based Technique of a Self-Adapting Autoencoder and Internal Validation Indexes.

In [22]:
# Hyperparameter values
k_values = [5, 8, 10, 12, 15]
maxIter_values = [20, 50]
initMode_values = ["k-means||", "random"]
initSteps_values = [2, 5, 10]

Silhouette evaluator from PySpark is used as evaluator for Hyperparameter tuning as it has efficient evaluation directly on Spark DataFrames without collecting large datasets into driver memory

In [23]:
evaluator = ClusteringEvaluator(
    featuresCol="features",
    predictionCol="cluster",
    metricName="silhouette",
    distanceMeasure="squaredEuclidean"
)

For every iteration, we will:

1. Set the KMeans model parameters using the current hyperparameter combination
2. Train the model to the dataset
3. Generate cluster predictions
4. Evaluate the model using the Silhouette Score
5. Record the hyperparameter combination and evaluation result
6. Keep track of the best-performing model and its corresponding hyperparameters based on the highest Silhouette Score

In [24]:
results = []
best_score = -1
best_model = None
best_params = None

for k in k_values:
    for maxIter in maxIter_values:
        for initMode in initMode_values:
            for initSteps in initSteps_values:

                kmeans = KMeans(
                    featuresCol="features",
                    predictionCol="cluster",
                    k=k,
                    maxIter=maxIter,
                    initMode=initMode,
                    initSteps=initSteps,
                    seed=42
                )

                model = kmeans.fit(kmeans_df)

                predictions = model.transform(kmeans_df)

                silhouette = evaluator.evaluate(predictions)

                results.append({
                    "k": k,
                    "maxIter": maxIter,
                    "initMode": initMode,
                    "initSteps": initSteps,
                    "silhouette": silhouette
                })

                print(
                    f"k={k:2d}, "
                    f"maxIter={maxIter:3d}, "
                    f"initMode={initMode:10s}, "
                    f"initSteps={initSteps:2d}, "
                    f"Silhouette={silhouette:.4f}"
                )

                if silhouette > best_score:
                    best_score = silhouette
                    best_model = model
                    best_params = {
                        "k": k,
                        "maxIter": maxIter,
                        "initMode": initMode,
                        "initSteps": initSteps
                    }

print("\nBest Hyperparameters")
print(best_params)
print(f"Best Silhouette Score: {best_score:.4f}")

k= 5, maxIter= 20, initMode=k-means|| , initSteps= 2, Silhouette=0.3414


k= 5, maxIter= 20, initMode=k-means|| , initSteps= 5, Silhouette=0.2805


k= 5, maxIter= 20, initMode=k-means|| , initSteps=10, Silhouette=0.2788


k= 5, maxIter= 20, initMode=random    , initSteps= 2, Silhouette=0.2918


k= 5, maxIter= 20, initMode=random    , initSteps= 5, Silhouette=0.3024


...

k=15, maxIter= 50, initMode=k-means|| , initSteps=10, Silhouette=0.3507


k=15, maxIter= 50, initMode=random    , initSteps= 2, Silhouette=0.1865


k=15, maxIter= 50, initMode=random    , initSteps= 5, Silhouette=0.1865


[Stage 10186:>                                                      (0 + 4) / 4]

k=15, maxIter= 50, initMode=random    , initSteps=10, Silhouette=0.1865

Best Hyperparameters
{'k': 5, 'maxIter': 50, 'initMode': 'k-means||', 'initSteps': 5}
Best Silhouette Score: 0.4598


## Train the Final Model

The best parameter is

{'k': 5, 'maxIter': 50, 'initMode': 'k-means||', 'initSteps': 5}

Train the final model with the best parameter

In [25]:
final_kmeans = KMeans(
    featuresCol="features",
    predictionCol="cluster",
    **best_params
)

In [26]:
final_model = final_kmeans.fit(kmeans_df)

In [27]:
clustered_products = final_model.transform(kmeans_df)

Using Sklearn's Silhouette Score, Davies-Bouldin Index (DBI), Calinski-Harabasz Index (CHI), evaluate the model

Silhouette Score used a 20,000-sample subset to reduce computation from pairwise distance calculations while maintaining a reliable estimate. DBI and CHI were calculated on the full dataset due to their lower computational cost.

In [28]:
pdf = clustered_products.select("features", "cluster").toPandas()

X = np.vstack(pdf["features"].apply(lambda x: x.toArray()))
labels = pdf["cluster"].values

silhouette = silhouette_score(
    X,
    labels,
    sample_size=20000,
    random_state=42
)

dbi = davies_bouldin_score(X, labels)
chi = calinski_harabasz_score(X, labels)

print(f"Silhouette Score:       {silhouette:.4f}")
print(f"Davies-Bouldin Index:   {dbi:.4f}")
print(f"Calinski-Harabasz Index:{chi:.4f}")

Silhouette Score:       0.2228
Davies-Bouldin Index:   1.3689
Calinski-Harabasz Index:140629.8086


Check the rate for each condition in the clusters

In [29]:
clustered_products = clustered_products.withColumn(
    "total_conditions",
    F.col("cond_good") +
    F.col("cond_new") +
    F.col("cond_like_new") +
    F.col("cond_fair") +
    F.col("cond_poor") +
    F.col("cond_unknown")
)

In [30]:
clustered_products = (
    clustered_products
    .withColumn(
        "new_rate",
        F.when(
            F.col("total_conditions") > 0,
            F.col("cond_new") / F.col("total_conditions")
        ).otherwise(0)
    )
    .withColumn(
        "like_new_rate",
        F.when(
            F.col("total_conditions") > 0,
            F.col("cond_like_new") / F.col("total_conditions")
        ).otherwise(0)
    )
    .withColumn(
        "good_rate",
        F.when(
            F.col("total_conditions") > 0,
            F.col("cond_good") / F.col("total_conditions")
        ).otherwise(0)
    )
    .withColumn(
        "fair_rate",
        F.when(
            F.col("total_conditions") > 0,
            F.col("cond_fair") / F.col("total_conditions")
        ).otherwise(0)
    )
    .withColumn(
        "poor_rate",
        F.when(
            F.col("total_conditions") > 0,
            F.col("cond_poor") / F.col("total_conditions")
        ).otherwise(0)
    )
    .withColumn(
        "unknown_rate",
        F.when(
            F.col("total_conditions") > 0,
            F.col("cond_unknown") / F.col("total_conditions")
        ).otherwise(0)
    )
)

Check the summary per each cluster

In [31]:
cluster_summary = (
    clustered_products
    .groupBy("cluster")
    .agg(
        # Cluster size
        F.count("*").alias("num_products"),

        # Customer engagement
        F.round(F.avg("views"), 2).alias("avg_views"),
        F.round(F.avg("likes"), 2).alias("avg_likes"),
        F.round(F.avg("cart"), 2).alias("avg_cart"),
        F.round(F.avg("buy_comp"), 2).alias("avg_purchases"),

        # Customer reach
        F.round(F.avg("unique_users"), 2).alias("avg_unique_users"),
        F.round(F.avg("unique_sessions"), 2).alias("avg_unique_sessions"),

        # Product characteristics
        F.round(F.avg("avg_price"), 2).alias("avg_price"),

        # Condition profile
        F.round(F.avg("new_rate"), 3).alias("pct_new"),
        F.round(F.avg("like_new_rate"), 3).alias("pct_like_new"),
        F.round(F.avg("good_rate"), 3).alias("pct_good"),
        F.round(F.avg("fair_rate"), 3).alias("pct_fair"),
        F.round(F.avg("poor_rate"), 3).alias("pct_poor"),
        F.round(F.avg("unknown_rate"), 3).alias("pct_unknown")
    )
    .orderBy("cluster")
)

cluster_summary.show(truncate=False)

[Stage 10403:>                                                      (0 + 4) / 4]

+-------+------------+---------+---------+--------+-------------+----------------+-------------------+---------+-------+------------+--------+--------+--------+-----------+
|cluster|num_products|avg_views|avg_likes|avg_cart|avg_purchases|avg_unique_users|avg_unique_sessions|avg_price|pct_new|pct_like_new|pct_good|pct_fair|pct_poor|pct_unknown|
+-------+------------+---------+---------+--------+-------------+----------------+-------------------+---------+-------+------------+--------+--------+--------+-----------+
|0      |74044       |2.26     |0.24     |0.02    |0.0          |2.51            |2.53               |49.9     |0.278  |0.288       |0.413   |0.02    |0.002   |0.0        |
|1      |287903      |1.74     |0.19     |0.02    |0.0          |1.94            |1.96               |51.33    |0.37   |0.238       |0.364   |0.025   |0.003   |0.0        |
|2      |71417       |18.57    |2.42     |0.21    |0.01         |20.67           |21.25              |56.97    |0.35   |0.253       |0.